# **2. Train and Test ML algorithm in Materials - Supervised Learning**

**Load and prepare a pre-featurized dataset**

In [52]:
# Install libraries to use matminer.
!pip install --upgrade matplotlib==3.8.0 -q
!pip install --upgrade pyyaml six matminer[citrine] citrination-client pymatgen -q
!pip install --upgrade pandas==2.2.2 -q

In [53]:
from matminer.datasets.convenience_loaders import load_elastic_tensor

df = load_elastic_tensor()  # load the dataset in a pandas DataFrame object
df.columns # check columns

Index(['material_id', 'formula', 'nsites', 'space_group', 'volume',
       'structure', 'elastic_anisotropy', 'G_Reuss', 'G_VRH', 'G_Voigt',
       'K_Reuss', 'K_VRH', 'K_Voigt', 'poisson_ratio', 'compliance_tensor',
       'elastic_tensor', 'elastic_tensor_original'],
      dtype='object')

In [54]:
# Removing unneeded columns from the data set
# Not all data is required for modeling.
unwanted_columns = ["volume", "nsites", "compliance_tensor", "elastic_tensor",
                    "elastic_tensor_original", "K_Voigt", "G_Voigt", "K_Reuss", "G_Reuss"]
df = df.drop(unwanted_columns, axis=1)
df.head()

,material_id,formula,space_group,structure,elastic_anisotropy,G_VRH,K_VRH,poisson_ratio
0,mp-10003,Nb4CoSi,124,"[[0.94814328 2.07280467 2.5112 ] Nb, [5.273...",0.030688,97.141604,194.268884,0.285701
1,mp-10010,Al(CoSi)2,164,"[[0. 0. 0.] Al, [1.96639263 1.13529553 0.75278...",0.266910,96.252006,175.449907,0.268105
2,mp-10015,SiOs,221,"[[1.480346 1.480346 1.480346] Si, [0. 0. 0.] Os]",0.756489,130.112955,295.077545,0.307780
3,mp-10021,Ga,63,"[[0. 1.09045794 0.84078375] Ga, [0. ...",2.376805,15.101901,49.130670,0.360593
4,mp-10025,SiRu2,62,"[[1.0094265 4.24771709 2.9955487 ] Si, [3.028...",0.196930,101.947798,256.768081,0.324682


In [55]:
# Add composition-based features
# A major class of featurizers available in matminer uses the chemical composition to featurize the input data.
# Let's add some composition based features to our DataFrame.

# First step : Using the conversions Featurizers in matminer to turn a String composition (our 'formula' column from before) into a pymatgen Composition.
from matminer.featurizers.conversions import StrToComposition

df = StrToComposition().featurize_dataframe(df, "formula")
df.head() # It will be possible to confirm that a new composition column is formed.

StrToComposition:   0%|          | 0/1181 [00:00<?, ?it/s]

,material_id,formula,space_group,structure,elastic_anisotropy,G_VRH,K_VRH,poisson_ratio,composition
0,mp-10003,Nb4CoSi,124,"[[0.94814328 2.07280467 2.5112 ] Nb, [5.273...",0.030688,97.141604,194.268884,0.285701,"(Nb, Co, Si)"
1,mp-10010,Al(CoSi)2,164,"[[0. 0. 0.] Al, [1.96639263 1.13529553 0.75278...",0.266910,96.252006,175.449907,0.268105,"(Al, Co, Si)"
2,mp-10015,SiOs,221,"[[1.480346 1.480346 1.480346] Si, [0. 0. 0.] Os]",0.756489,130.112955,295.077545,0.307780,"(Si, Os)"
3,mp-10021,Ga,63,"[[0. 1.09045794 0.84078375] Ga, [0. ...",2.376805,15.101901,49.130670,0.360593,(Ga)
4,mp-10025,SiRu2,62,"[[1.0094265 4.24771709 2.9955487 ] Si, [3.028...",0.196930,101.947798,256.768081,0.324682,"(Si, Ru)"


In [56]:
# Second step : Using one of the featurizers in matminer to add a suite of descriptors to the DataFrame.
from matminer.featurizers.composition import ElementProperty

ep_feat = ElementProperty.from_preset(preset_name="magpie")
df = ep_feat.featurize_dataframe(df, col_id="composition")  # input the "composition" column to the featurizer
df.head() # It can be seen that many composition-related features are newly created in the data.

/usr/local/lib/python3.12/dist-packages/matminer/utils/data.py:326: UserWarning: MagpieData(impute_nan=False):
In a future release, impute_nan will be set to True by default.
                    This means that features that are missing or are NaNs for elements
                    from the data source will be replaced by the average of that value
                    over the available elements.
                    This avoids NaNs after featurization that are often replaced by
                    dataset-dependent averages.
  warnings.warn(f"{self.__class__.__name__}(impute_nan=False):\n" + IMPUTE_NAN_WARNING)


ElementProperty:   0%|          | 0/1181 [00:00<?, ?it/s]

,material_id,formula,space_group,structure,elastic_anisotropy,G_VRH,K_VRH,poisson_ratio,composition,MagpieData minimum Number,...,MagpieData range GSmagmom,MagpieData mean GSmagmom,MagpieData avg_dev GSmagmom,MagpieData mode GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber
0,mp-10003,Nb4CoSi,124,"[[0.94814328 2.07280467 2.5112 ] Nb, [5.273...",0.030688,97.141604,194.268884,0.285701,"(Nb, Co, Si)",14.0,...,1.548471,0.258079,0.430131,0.0,194.0,229.0,35.0,222.833333,9.611111,229.0
1,mp-10010,Al(CoSi)2,164,"[[0. 0. 0.] Al, [1.96639263 1.13529553 0.75278...",0.266910,96.252006,175.449907,0.268105,"(Al, Co, Si)",13.0,...,1.548471,0.619388,0.743266,0.0,194.0,227.0,33.0,213.400000,15.520000,194.0
2,mp-10015,SiOs,221,"[[1.480346 1.480346 1.480346] Si, [0. 0. 0.] Os]",0.756489,130.112955,295.077545,0.307780,"(Si, Os)",14.0,...,0.000000,0.000000,0.000000,0.0,194.0,227.0,33.0,210.500000,16.500000,194.0
3,mp-10021,Ga,63,"[[0. 1.09045794 0.84078375] Ga, [0. ...",2.376805,15.101901,49.130670,0.360593,(Ga),31.0,...,0.000000,0.000000,0.000000,0.0,64.0,64.0,0.0,64.000000,0.000000,64.0
4,mp-10025,SiRu2,62,"[[1.0094265 4.24771709 2.9955487 ] Si, [3.028...",0.196930,101.947798,256.768081,0.324682,"(Si, Ru)",14.0,...,0.000000,0.000000,0.000000,0.0,194.0,227.0,33.0,205.000000,14.666667,194.0


In [57]:
# Add more composition-based features
# There are many more Composition based featurizers apart from ElementProperty that are available in the matminer.featurizers.composition.
# Let's try the ElectronegativityDiff featurizer which requires knowing the oxidation state of the various elements in the Composition.

from matminer.featurizers.conversions import CompositionToOxidComposition
from matminer.featurizers.composition import OxidationStates

df = CompositionToOxidComposition().featurize_dataframe(df, "composition") # composition

os_feat = OxidationStates()
df = os_feat.featurize_dataframe(df, "composition_oxid") # composition_oxid // add oxidation states
df.head()

CompositionToOxidComposition:   0%|          | 0/1181 [00:00<?, ?it/s]

OxidationStates:   0%|          | 0/1181 [00:00<?, ?it/s]

,material_id,formula,space_group,structure,elastic_anisotropy,G_VRH,K_VRH,poisson_ratio,composition,MagpieData minimum Number,...,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber,composition_oxid,minimum oxidation state,maximum oxidation state,range oxidation state,std_dev oxidation state
0,mp-10003,Nb4CoSi,124,"[[0.94814328 2.07280467 2.5112 ] Nb, [5.273...",0.030688,97.141604,194.268884,0.285701,"(Nb, Co, Si)",14.0,...,229.0,35.0,222.833333,9.611111,229.0,"(Nb0+, Co0+, Si0+)",0,0,0,0.000000
1,mp-10010,Al(CoSi)2,164,"[[0. 0. 0.] Al, [1.96639263 1.13529553 0.75278...",0.266910,96.252006,175.449907,0.268105,"(Al, Co, Si)",13.0,...,227.0,33.0,213.400000,15.520000,194.0,"(Al3+, Co2+, Co3+, Si4-)",-4,3,7,3.872983
2,mp-10015,SiOs,221,"[[1.480346 1.480346 1.480346] Si, [0. 0. 0.] Os]",0.756489,130.112955,295.077545,0.307780,"(Si, Os)",14.0,...,227.0,33.0,210.500000,16.500000,194.0,"(Si4-, Os4+)",-4,4,8,5.656854
3,mp-10021,Ga,63,"[[0. 1.09045794 0.84078375] Ga, [0. ...",2.376805,15.101901,49.130670,0.360593,(Ga),31.0,...,64.0,0.0,64.000000,0.000000,64.0,(Ga0+),0,0,0,0.000000
4,mp-10025,SiRu2,62,"[[1.0094265 4.24771709 2.9955487 ] Si, [3.028...",0.196930,101.947798,256.768081,0.324682,"(Si, Ru)",14.0,...,227.0,33.0,205.000000,14.666667,194.0,"(Si4-, Ru2+)",-4,2,6,4.242641


In [58]:
# Add some structure based features
from matminer.featurizers.structure import DensityFeatures

df_feat = DensityFeatures()
df = df_feat.featurize_dataframe(df, "structure")  # input the 'structure' column to the featurizer
df.head()

DensityFeatures:   0%|          | 0/1181 [00:00<?, ?it/s]

,material_id,formula,space_group,structure,elastic_anisotropy,G_VRH,K_VRH,poisson_ratio,composition,MagpieData minimum Number,...,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber,composition_oxid,minimum oxidation state,maximum oxidation state,range oxidation state,std_dev oxidation state,density,vpa,packing fraction
0,mp-10003,Nb4CoSi,124,"[[0.94814328 2.07280467 2.5112 ] Nb, [5.273...",0.030688,97.141604,194.268884,0.285701,"(Nb, Co, Si)",14.0,...,9.611111,229.0,"(Nb0+, Co0+, Si0+)",0,0,0,0.000000,7.834556,16.201654,0.688834
1,mp-10010,Al(CoSi)2,164,"[[0. 0. 0.] Al, [1.96639263 1.13529553 0.75278...",0.266910,96.252006,175.449907,0.268105,"(Al, Co, Si)",13.0,...,15.520000,194.0,"(Al3+, Co2+, Co3+, Si4-)",-4,3,7,3.872983,5.384968,12.397466,0.644386
2,mp-10015,SiOs,221,"[[1.480346 1.480346 1.480346] Si, [0. 0. 0.] Os]",0.756489,130.112955,295.077545,0.307780,"(Si, Os)",14.0,...,16.500000,194.0,"(Si4-, Os4+)",-4,4,8,5.656854,13.968635,12.976265,0.569426
3,mp-10021,Ga,63,"[[0. 1.09045794 0.84078375] Ga, [0. ...",2.376805,15.101901,49.130670,0.360593,(Ga),31.0,...,0.000000,64.0,(Ga0+),0,0,0,0.000000,6.036267,19.180359,0.479802
4,mp-10025,SiRu2,62,"[[1.0094265 4.24771709 2.9955487 ] Si, [3.028...",0.196930,101.947798,256.768081,0.324682,"(Si, Ru)",14.0,...,14.666667,194.0,"(Si4-, Ru2+)",-4,2,6,4.242641,9.539514,13.358418,0.598395


**Step1: Train ML algorithm with Bulk Modulus (Kvrh) data**

In [59]:
# y - target property(bulk modulus -> 'K_VRH'), X - remove data of string type (computer cannot accept) and initially given elastic data
y = df['K_VRH'].values
excluded = ["G_VRH", "K_VRH", "elastic_anisotropy", "formula", "material_id",
            "poisson_ratio", "structure", "composition", "composition_oxid", "space_group"]
X = df.drop(excluded, axis=1)
print("There are {} possible descriptors:\n\n{}".format(X.shape[1], X.columns.values))

There are 139 possible descriptors:

['MagpieData minimum Number' 'MagpieData maximum Number'
 'MagpieData range Number' 'MagpieData mean Number'
 'MagpieData avg_dev Number' 'MagpieData mode Number'
 'MagpieData minimum MendeleevNumber' 'MagpieData maximum MendeleevNumber'
 'MagpieData range MendeleevNumber' 'MagpieData mean MendeleevNumber'
 'MagpieData avg_dev MendeleevNumber' 'MagpieData mode MendeleevNumber'
 'MagpieData minimum AtomicWeight' 'MagpieData maximum AtomicWeight'
 'MagpieData range AtomicWeight' 'MagpieData mean AtomicWeight'
 'MagpieData avg_dev AtomicWeight' 'MagpieData mode AtomicWeight'
 'MagpieData minimum MeltingT' 'MagpieData maximum MeltingT'
 'MagpieData range MeltingT' 'MagpieData mean MeltingT'
 'MagpieData avg_dev MeltingT' 'MagpieData mode MeltingT'
 'MagpieData minimum Column' 'MagpieData maximum Column'
 'MagpieData range Column' 'MagpieData mean Column'
 'MagpieData avg_dev Column' 'MagpieData mode Column'
 'MagpieData minimum Row' 'MagpieData maximum 

Use Linear Regression model from Scikit-learn

In [60]:
#Linear regression is a linear approach for modelling the relationship between a scalar target value and one or more explanatory variables (also known as dependent and independent variables).
#For more than one, the process is called multiple linear regression.
#This term is distinct from multivariate linear regression, where multiple correlated dependent variables are predicted, rather than a single scalar variable.
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

# use 'LinearRegression'
lr = LinearRegression()
lr.fit(X, y)

# get fit statistics
print('training R2 = ' + str(round(lr.score(X, y), 3)))
print('training RMSE = %.3f' % np.sqrt(mean_squared_error(y_true=y, y_pred=lr.predict(X))))

training R2 = 0.926
training RMSE = 19.761


In [61]:
# To prevent overfitting, we need to check the cross-validation score
from sklearn.model_selection import KFold, cross_val_score

# Use 'KFold'(10-fold) cross validation (90% training, 10% test), shuffle the data before splitting
crossvalidation = KFold(n_splits=10, shuffle=True, random_state=1)

# compute cross validation scores('cross_val_score') for linear regression model
scores = cross_val_score(lr, X, y, scoring='neg_mean_squared_error', cv=crossvalidation, n_jobs=1)
rmse_scores = [np.sqrt(abs(s)) for s in scores]
r2_scores = cross_val_score(lr, X, y, scoring='r2', cv=crossvalidation, n_jobs=1)

#cv: the number of fold in cross-validation, n jobs: the number of CPU in parallel computational simulation
print('Cross-validation results:')
print('Folds: %i, mean R2: %.3f' % (len(scores), np.mean(np.abs(r2_scores))))
print('Folds: %i, mean RMSE: %.3f' % (len(scores), np.mean(np.abs(rmse_scores))))

Cross-validation results:
Folds: 10, mean R2: 0.900
Folds: 10, mean RMSE: 22.642


Use Random Forest model


In [62]:
#Random Forest Regression is a supervised learning algorithm that uses ensemble learning method for regression.
#Ensemble learning method is a technique that combines predictions from multiple machine learning algorithms to make a more accurate prediction than a single model.
#the trees run in parallel with no interaction amongst them. A Random Forest operates by constructing several decision trees during training time and outputting
#the mean of the classes as the prediction of all the trees.
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=50, random_state=1)

rf.fit(X, y)
print('training R2 = ' + str(round(rf.score(X, y), 3)))
print('training RMSE = %.3f' % np.sqrt(mean_squared_error(y_true=y, y_pred=rf.predict(X))))

training R2 = 0.989
training RMSE = 7.753


In [63]:
# compute cross validation scores('cross_val_score') for random forest model
r2_scores = cross_val_score(rf, X, y, scoring='r2', cv=crossvalidation, n_jobs=-1)
scores = cross_val_score(rf, X, y, scoring='neg_mean_squared_error', cv=crossvalidation, n_jobs=-1)
rmse_scores = [np.sqrt(abs(s)) for s in scores]

print('Cross-validation results:')
print('Folds: %i, mean R2: %.3f' % (len(scores), np.mean(np.abs(r2_scores))))
print('Folds: %i, mean RMSE: %.3f' % (len(scores), np.mean(np.abs(rmse_scores))))

Cross-validation results:
Folds: 10, mean R2: 0.925
Folds: 10, mean RMSE: 19.083


**Step2: Test with Random Forest model**

In [64]:
from sklearn.model_selection import train_test_split
X['formula'] = df['formula']
# It is not used for training, but it can be necessary when you want to confirm the formula of the sample
# If we don't do this process, we can't track the formula data after tain/test split('train_test_split')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
train_formula = X_train['formula']
X_train = X_train.drop('formula', axis=1)
test_formula = X_test['formula']
X_test = X_test.drop('formula', axis=1)

# Use Random Forest model(RandomForestRegressor)
rf_reg = RandomForestRegressor(n_estimators=50, random_state=1)
rf_reg.fit(X_train, y_train)

# get fit statistics
print('training R2 = ' + str(round(rf_reg.score(X_train, y_train), 3)))
print('training RMSE = %.3f' % np.sqrt(mean_squared_error(y_true=y_train, y_pred=rf_reg.predict(X_train))))
print('test R2 = ' + str(round(rf_reg.score(X_test, y_test), 3)))
print('test RMSE = %.3f' % np.sqrt(mean_squared_error(y_true=y_test, y_pred=rf_reg.predict(X_test))))

training R2 = 0.987
training RMSE = 8.260
test R2 = 0.942
test RMSE = 16.949


In [65]:
df_sample = df.loc[[0], :]
excluded = ["G_VRH", "K_VRH", "elastic_anisotropy", "formula", "material_id",
            "poisson_ratio", "structure", "composition", "composition_oxid", "space_group"]
Z = df_sample.drop(excluded, axis=1)
y_pred=rf_reg.predict(Z)

y_pred

array([196.88977714])

We can optimize various parameters for machine learning algorithm by using AutoML task. In this section, we will performn AutoML through "FLAML(Fast and Lightweight AutoML".

In [66]:
!pip install flaml -q

from flaml import AutoML
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Create and train the 'AutoML' model
automl = AutoML()
automl.fit(X_train, y_train, task="regression", estimator_list=["rf"], time_budget=60)

# get fit statistics
print('training R2 = ' + str(round(automl.score(X_train, y_train), 3)))
print('training RMSE = %.3f' % np.sqrt(mean_squared_error(y_true=y_train, y_pred=automl.predict(X_train))))

[flaml.automl.logger: 09-27 11:52:20] {1752} INFO - task = regression
[flaml.automl.logger: 09-27 11:52:20] {1763} INFO - Evaluation method: cv
[flaml.automl.logger: 09-27 11:52:20] {1862} INFO - Minimizing error metric: 1-r2
[flaml.automl.logger: 09-27 11:52:20] {1979} INFO - List of ML learners in AutoML Run: ['rf']
[flaml.automl.logger: 09-27 11:52:20] {2282} INFO - iteration 0, current learner rf
[flaml.automl.logger: 09-27 11:52:20] {2417} INFO - Estimated sufficient time budget=4185s. Estimated necessary time budget=4s.
[flaml.automl.logger: 09-27 11:52:20] {2466} INFO -  at 0.5s,	estimator rf's best error=0.3063,	best estimator rf's best error=0.3063
[flaml.automl.logger: 09-27 11:52:20] {2282} INFO - iteration 1, current learner rf
[flaml.automl.logger: 09-27 11:52:21] {2466} INFO -  at 1.0s,	estimator rf's best error=0.1893,	best estimator rf's best error=0.1893
[flaml.automl.logger: 09-27 11:52:21] {2282} INFO - iteration 2, current learner rf
[flaml.automl.logger: 09-27 11:5

In [67]:
y_pred_automl = automl.predict(Z)
print(f"Best config: {automl.best_config}")
print('test R2 = ' + str(round(automl.score(X_test, y_test), 3)))
print('test RMSE = %.3f' % np.sqrt(mean_squared_error(y_true=y_test, y_pred=automl.predict(X_test))))

y_pred_automl

Best config: {'n_estimators': 73, 'max_features': 0.48911663396268434, 'max_leaves': 471}
test R2 = 0.943
test RMSE = 16.843


array([197.19035935])

**Predict a new material not contatined in the dataframe**

In [68]:
!pip install mp_api -q

from mp_api.client import MPRester
import pandas as pd

# Replace "your_api_key_here" with your own API key from the Materials Project
mpr = MPRester("CiInKG9AGVmp2w8U1vDl9U2KI4hK9dDF")
list_of_available_fields = mpr.materials.summary.available_fields

# Search for materials with formula Fe2O3 and retrieve all available fields
docs = mpr.materials.summary.search(formula="Fe2O3", fields=list_of_available_fields)
results = [doc.dict() for doc in docs]
df_full = pd.DataFrame(results)

df2 = df_full.loc[:, ["formula_pretty", "material_id", "formation_energy_per_atom", "structure"]]
sdf = df2.sort_values(by="formation_energy_per_atom")
sdf.head()

Retrieving SummaryDoc documents:   0%|          | 0/26 [00:00<?, ?it/s]

,formula_pretty,material_id,formation_energy_per_atom,structure
12,Fe2O3,mp-19770,-1.707091,"{'@module': 'pymatgen.core.structure', '@class..."
13,Fe2O3,mp-565814,-1.635792,"{'@module': 'pymatgen.core.structure', '@class..."
18,Fe2O3,mp-715572,-1.629914,"{'@module': 'pymatgen.core.structure', '@class..."
15,Fe2O3,mp-1356129,-1.622432,"{'@module': 'pymatgen.core.structure', '@class..."
20,Fe2O3,mp-1178392,-1.594236,"{'@module': 'pymatgen.core.structure', '@class..."


In [69]:
#extracting wanted material data row
df3 = df2[df2["material_id"] == "mp-19770"]
df3

,formula_pretty,material_id,formation_energy_per_atom,structure
12,Fe2O3,mp-19770,-1.707091,"{'@module': 'pymatgen.core.structure', '@class..."


In [70]:
from matminer.featurizers.conversions import StrToComposition
from matminer.featurizers.composition import ElementProperty
from matminer.featurizers.conversions import CompositionToOxidComposition
from matminer.featurizers.composition import OxidationStates
from matminer.featurizers.structure import DensityFeatures
from pymatgen.core.structure import Structure # import the Structure class from pymatgen

df3 = StrToComposition().featurize_dataframe(df3, "formula_pretty")

ep_feat = ElementProperty.from_preset(preset_name="magpie")
df3 = ep_feat.featurize_dataframe(df3, col_id="composition")  # input the "composition" column to the featurizer

df3 = CompositionToOxidComposition().featurize_dataframe(df3, "composition") # compositon -> composition_oxid // add oxidation states

os_feat = OxidationStates()
df3 = os_feat.featurize_dataframe(df3, "composition_oxid")

# Convert dictionaries in "structure" column to pymatgen Structure objects
df3["structure"] = df3["structure"].apply(lambda x: Structure.from_dict(x))

df3_feat = DensityFeatures()
df3 = df3_feat.featurize_dataframe(df3, "structure")  # input the structure column to the featurizer

StrToComposition:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/matminer/utils/data.py:326: UserWarning: MagpieData(impute_nan=False):
In a future release, impute_nan will be set to True by default.
                    This means that features that are missing or are NaNs for elements
                    from the data source will be replaced by the average of that value
                    over the available elements.
                    This avoids NaNs after featurization that are often replaced by
                    dataset-dependent averages.
  warnings.warn(f"{self.__class__.__name__}(impute_nan=False):\n" + IMPUTE_NAN_WARNING)


ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

CompositionToOxidComposition:   0%|          | 0/1 [00:00<?, ?it/s]

OxidationStates:   0%|          | 0/1 [00:00<?, ?it/s]

DensityFeatures:   0%|          | 0/1 [00:00<?, ?it/s]

In [71]:
excluded = ["material_id", "formation_energy_per_atom", "formula_pretty", "composition", "composition_oxid", "structure"]
Z = df3.drop(excluded, axis=1)
print("There are {} possible descriptors:\n\n{}".format(Z.shape[1], Z.columns.values))

There are 139 possible descriptors:

['MagpieData minimum Number' 'MagpieData maximum Number'
 'MagpieData range Number' 'MagpieData mean Number'
 'MagpieData avg_dev Number' 'MagpieData mode Number'
 'MagpieData minimum MendeleevNumber' 'MagpieData maximum MendeleevNumber'
 'MagpieData range MendeleevNumber' 'MagpieData mean MendeleevNumber'
 'MagpieData avg_dev MendeleevNumber' 'MagpieData mode MendeleevNumber'
 'MagpieData minimum AtomicWeight' 'MagpieData maximum AtomicWeight'
 'MagpieData range AtomicWeight' 'MagpieData mean AtomicWeight'
 'MagpieData avg_dev AtomicWeight' 'MagpieData mode AtomicWeight'
 'MagpieData minimum MeltingT' 'MagpieData maximum MeltingT'
 'MagpieData range MeltingT' 'MagpieData mean MeltingT'
 'MagpieData avg_dev MeltingT' 'MagpieData mode MeltingT'
 'MagpieData minimum Column' 'MagpieData maximum Column'
 'MagpieData range Column' 'MagpieData mean Column'
 'MagpieData avg_dev Column' 'MagpieData mode Column'
 'MagpieData minimum Row' 'MagpieData maximum 

In [72]:
Z.columns

Index(['MagpieData minimum Number', 'MagpieData maximum Number',
       'MagpieData range Number', 'MagpieData mean Number',
       'MagpieData avg_dev Number', 'MagpieData mode Number',
       'MagpieData minimum MendeleevNumber',
       'MagpieData maximum MendeleevNumber',
       'MagpieData range MendeleevNumber', 'MagpieData mean MendeleevNumber',
       ...
       'MagpieData mean SpaceGroupNumber',
       'MagpieData avg_dev SpaceGroupNumber',
       'MagpieData mode SpaceGroupNumber', 'minimum oxidation state',
       'maximum oxidation state', 'range oxidation state',
       'std_dev oxidation state', 'density', 'vpa', 'packing fraction'],
      dtype='object', length=139)

In [73]:
y_pred2 = rf_reg.predict(Z)
y_pred2

array([183.96479805])

In [74]:
y_pred_automl2=automl.predict(Z)
y_pred_automl2

array([179.4215247])

# **3. Train and Test ML algorithm in Materials - Unsupervised Learning**

In [75]:
!pip install matminer[citrine] -q # install matminer library
!pip install pyyaml -q
!pip install pymatgen -q

In [79]:
# If you want to use Google Drive, use the following code to mount Google Drive

from google.colab import drive
import os
import pandas as pd

drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Colab/')
df = pd.read_excel('/content/drive/MyDrive/modulus.xlsx')

Mounted at /content/drive


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab/'

In [77]:
# If you want to upload files directly, use the following code for a simple file upload
import pandas as pd

df = pd.read_excel('/content/modulus.xlsx')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/modulus.xlsx'

In [ ]:
excluded = ["material_id"]
X = df.drop(excluded, axis=1)

In [ ]:
#The agglomerative clustering is the most common type of hierarchical clustering used to group objects in clusters based on their similarity.
#It’s also known as AGNES (Agglomerative Nesting). The algorithm starts by treating each object as a singleton cluster.
#Next, pairs of clusters are successively merged until all clusters have been merged into one big cluster containing all objects.
import time as time
import matplotlib.pyplot as plt
import numpy as np


from sklearn.cluster import AgglomerativeClustering
import sklearn.datasets
#The method of merging two clusters involves combining the two clusters that result in the smallest increase in the variance within all clusters.(Ward)
print("Compute unstructured hierarchical clustering...")
st = time.time()
ward = AgglomerativeClustering(n_clusters=6, linkage="ward").fit(X)
elapsed_time = time.time() - st
group = ward.labels_
print(f"Elapsed time: {elapsed_time:.2f}s")
print(f"Number of points: {group.size}")

In [ ]:
group

In [ ]:
df['group'] = group.tolist()
df

In [ ]:
#AgglomerativeClustering --> many clusters --> connectivity constraints 필요 (only adjacent clusters can be merged together), through a connectivity matrix that defines for each sample the neighboring samples following a given structure of the data.
#For instance, in the swiss-roll example below, the connectivity constraints forbid the merging of points that are not adjacent on the swiss roll, and thus avoid forming clusters that extend across overlapping folds of the roll.
#The connectivity constraints are imposed via an connectivity matrix: a scipy sparse matrix that has elements only at the intersection of a row and a column with indices of the dataset that should be connected
from sklearn.neighbors import kneighbors_graph

connectivity = kneighbors_graph(X, n_neighbors=10, include_self=False)
print("Compute structured hierarchical clustering...")
st = time.time()
ward = AgglomerativeClustering(
    n_clusters=6, connectivity=connectivity, linkage="ward"
).fit(X)
elapsed_time = time.time() - st
group = ward.labels_
print(f"Elapsed time: {elapsed_time:.2f}s")
print(f"Number of points: {group.size}")

#Connectivity constraints and single, complete or average linkage can enhance the ‘rich getting richer’ aspect of agglomerative clustering, particularly so if they are built with sklearn.neighbors.kneighbors_graph.
#In the limit of a small number of clusters, they tend to give a few macroscopically occupied clusters and almost empty ones. (see the discussion in Agglomerative clustering with and without structure).
#Single linkage is the most brittle linkage option with regard to this issue.

In [ ]:
group

In [ ]:
df['group'] = group.tolist()

print(df)